# 第 6 课：从 `while` 循环到状态图——崩溃恢复、幂等键与时间旅行
### 《现代 AI 与 Agent 系统》Colab 实战实验 · 零配置即开即跑

> **为什么需要这节课？**
> 在第 3.5 课中，我们用几行 `while` 循环写出了一个能调工具的 Agent。但只要把它放到真实生产环境里跑一个 5 步任务，一个致命问题就会出现：**如果进程在第 4 步（比如刚刚调完扣款接口、还没来得及存盘）突然崩溃，重启后怎么办？**
> - 从头重跑？前 3 步的模型 Token 白花一遍，更糟的是**第 4 步的扣款会被重复执行两次**！
> - 怎么才能让 Agent 像现代软件一样：**崩了从断点接着跑、外部写操作绝不重复执行、甚至能退回第 3 步修改状态分叉重跑（Time-Travel）**？

---

### 实验方法论（OpenLabs 三步法）
本 Notebook 包含 **3 个递进的微型实验**。每个实验遵循：
1. **先写预测（Predict）**：在运行代码前，先凭直觉填好你预测的数字；
2. **动手实现与实测（Run）**：内置确定性录播替身模型（`RecordedLLM`，**没有 API Key 也能直接跑出精确数字**；如有 Key 也可切换真模型）；
3. **只用跑出来的数字下结论（Conclude）**：对比改动前后的浪费步数、重复扣款次数与账单差异。


In [ ]:
# [Cell 1: 环境初始化与替身模型/真实模型双模底座]
import copy
import hashlib
import json
import os
import sqlite3
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

# 可选：如果你想用真实大模型体验，在此填入 API Key（留空则默认使用零开销确定性录播模型）
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

@dataclass
class TripState:
    task_id: str
    step_index: int = 0
    flight_booked: Optional[str] = None
    hotel_booked: Optional[str] = None
    charged_amount: int = 0
    summary: Optional[str] = None
    history: List[str] = field(default_factory=list)

class PaymentGatewaySimulator:
    """模拟外部扣款网关：记录每一次真实发生的扣款请求。"""
    def __init__(self):
        self.charges: List[Dict[str, Any]] = []
        self.idempotency_store: Dict[str, Dict[str, Any]] = {}

    def charge_naive(self, task_id: str, amount: int) -> Dict[str, Any]:
        """无保护的直接扣款：每次调用都会真实扣钱！"""
        record = {"tx_id": f"tx_{len(self.charges) + 1}", "task_id": task_id, "amount": amount}
        self.charges.append(record)
        return record

    def charge_idempotent(self, idempotency_key: str, task_id: str, amount: int) -> Dict[str, Any]:
        """带幂等键的扣款：相同 idempotency_key 只会扣款一次，后续重试原样返回首次收据。"""
        if idempotency_key in self.idempotency_store:
            saved = self.idempotency_store[idempotency_key]
            if saved["amount"] != amount or saved["task_id"] != task_id:
                raise ValueError(f"IdempotencyKeyConflict: key={idempotency_key} reused with different payload!")
            return {**saved, "replayed": True}
        record = {
            "tx_id": f"tx_{len(self.charges) + 1}",
            "task_id": task_id,
            "amount": amount,
            "idempotency_key": idempotency_key,
            "replayed": False,
        }
        self.charges.append(record)
        self.idempotency_store[idempotency_key] = record
        return record

print("[OK] Cell 1 初始化完成！PaymentGatewaySimulator 与 TripState 已就绪。")


---
## 实验 1：`while` 循环 vs 显式状态图（崩溃之后从哪里接着跑？）

### 1. 为什么需要它？
假设一个订行程 Agent 需要依次完成 4 个节点：
1. `search_flight`（查航班，调 1 次模型）
2. `book_hotel`（订酒店，调 1 次模型）
3. `pay_order`（扣款 500 元，调外部支付接口 + 调 1 次模型）
4. `write_summary`（生成行程单，调 1 次模型）

如果进程在运行途中**随机崩溃**（例如网络闪断、容器 OOM、或者在刚扣完款还没存盘的那道缝隙崩溃），传统的 `while` 循环因为没有把每一步的中间状态存成**检查点（Checkpoint）**，只能从第 1 步重头再来。

### 2. 先写下你的预测
假设我们让这个 4 步任务跑 **100 次**，每次都在第 3 步（`pay_order` 刚执行完扣款、但还没进入第 4 步时）发生一次进程崩溃并重启完成：
- **预测 A**：普通 `while` 循环重跑完这 100 个任务，总共会调多少次模型？会发生多少次**重复扣款**？
- **预测 B**：只加了节点级 Checkpoint、但**没加幂等键**的状态图，在“刚扣完款、还没写入 Checkpoint”的缝隙崩溃，会发生多少次重复扣款？
- **预测 C**：同时具备 **Checkpoint + 幂等键（Idempotency Key）** 的状态图，总共调多少次模型？发生多少次重复扣款？


In [ ]:
# [Cell 2: 实验 1 — 对比 While 循环、仅 Checkpoint 状态图、Checkpoint + 幂等键状态图]

# 👉 第一步：在运行前填好你的预测数字
MY_PREDICTION_EXP1 = {
    "naive_while_extra_llm_calls": 300,      # 预测：普通 while 循环多浪费了多少次模型调用？
    "checkpoint_only_duplicate_charges": 100, # 预测：如果在“扣完款、存盘前”崩溃，仅靠 Checkpoint 会多扣几次款？
    "durable_idempotent_duplicate_charges": 0 # 预测：加上幂等键后，会多扣几次款？
}

def run_naive_while_with_crash(task_id: str, gateway: PaymentGatewaySimulator) -> Tuple[int, int]:
    """普通 while 循环：在第 3 步扣款后、完成前崩溃一次，重启后只能从头跑。"""
    llm_calls = 0
    for attempt in (1, 2):  # 第 1 次崩溃，第 2 次跑通
        # Step 1: search_flight
        llm_calls += 1
        # Step 2: book_hotel
        llm_calls += 1
        # Step 3: pay_order
        llm_calls += 1
        gateway.charge_naive(task_id, 500)
        if attempt == 1:
            # 💥 模拟在第 3 步刚扣完款时进程崩溃！内存状态全部丢失
            continue
        # Step 4: write_summary
        llm_calls += 1
    return llm_calls, len([c for c in gateway.charges if c["task_id"] == task_id])

# 🛠️ 作业任务 1：请补全带 Checkpoint + 幂等键的持久状态图恢复函数
def run_durable_graph_with_crash(task_id: str, gateway: PaymentGatewaySimulator, use_idempotency: bool = True) -> Tuple[int, int]:
    """
    显式状态图：
    - Step 1 和 Step 2 完成后已安全写入 checkpoint_db（因此崩溃后无需重跑 Step 1 & 2！）
    - 在 Step 3 中：扣款刚刚完成，但在写入 Step 3 的 checkpoint 之前进程崩溃（即著名的“做完还没存”缝隙）。
    - 重启后，状态图从 Step 2 的 checkpoint 恢复，重新进入 Step 3，然后完成 Step 4。
    """
    checkpoint_step = 0
    llm_calls = 0

    for attempt in (1, 2):
        # Step 1
        if checkpoint_step < 1:
            llm_calls += 1
            checkpoint_step = 1
        # Step 2
        if checkpoint_step < 2:
            llm_calls += 1
            checkpoint_step = 2
        # Step 3 (包含外部副作用：扣款 500 元)
        if checkpoint_step < 3:
            llm_calls += 1
            # TODO: 请根据 use_idempotency 参数选择调用 gateway.charge_idempotent 或 gateway.charge_naive
            # 提示：幂等键应当由 (task_id, step_name) 唯一确定，例如 f"{task_id}:step3_pay"
            if use_idempotency:
                # === YOUR CODE HERE (约 2 行代码) ===
                # 构造唯一幂等键 f"{task_id}:step3_pay" 并调用 gateway.charge_idempotent(...)
                idem_key = f"{task_id}:step3_pay"
                gateway.charge_idempotent(idem_key, task_id, 500)
                # ==================================
            else:
                gateway.charge_naive(task_id, 500)

            if attempt == 1:
                # 💥 模拟在“扣款已完成、但 checkpoint_step = 3 还没落盘”的瞬间崩溃！
                continue
            checkpoint_step = 3
        # Step 4
        if checkpoint_step < 4:
            llm_calls += 1
            checkpoint_step = 4

    tx_count = len([c for c in gateway.charges if c["task_id"] == task_id])
    return llm_calls, tx_count

# 运行 100 次对比实验
gw_naive = PaymentGatewaySimulator()
gw_cp_only = PaymentGatewaySimulator()
gw_durable = PaymentGatewaySimulator()

naive_calls = sum(run_naive_while_with_crash(f"t_{i}", gw_naive)[0] for i in range(100))
cp_only_calls = sum(run_durable_graph_with_crash(f"t_{i}", gw_cp_only, use_idempotency=False)[0] for i in range(100))
durable_calls = sum(run_durable_graph_with_crash(f"t_{i}", gw_durable, use_idempotency=True)[0] for i in range(100))

print("=== 实验 1 实测结果（100 个任务，均在 Step 3 存盘前崩溃 1 次）===")
print(f"1. 普通 while 循环       : 总模型调用 = {naive_calls} 次 (多浪费 {naive_calls - 400} 次) | 实际扣款笔数 = {len(gw_naive.charges)} 笔 (重复扣款 {len(gw_naive.charges) - 100} 次)")
print(f"2. 仅 Checkpoint 无幂等键: 总模型调用 = {cp_only_calls} 次 (多浪费 {cp_only_calls - 400} 次) | 实际扣款笔数 = {len(gw_cp_only.charges)} 笔 (重复扣款 {len(gw_cp_only.charges) - 100} 次)")
print(f"3. Checkpoint + 幂等键   : 总模型调用 = {durable_calls} 次 (多浪费 {durable_calls - 400} 次) | 实际扣款笔数 = {len(gw_durable.charges)} 笔 (重复扣款 {len(gw_durable.charges) - 100} 次)")


---
## 实验 2：并行分支写同一个状态字段——合并函数（Reducer）决定谁留下

### 1. 为什么需要它？
当状态图把任务拆给 3 个并行分支（比如同时查 3 家航空公司的报价），它们完成的时间先后是不确定的。如果 3 个分支都往状态的同一个字段 `quotes` 写结果：
- **直接覆盖（Overwrite）**：谁最后跑完，谁就把前面两个分支的结果覆盖掉；
- **简单列表追加（Append）**：如果某个分支因为网络抖动重试了一次，列表里就会出现重复数据，且完成顺序一变、列表顺序就变，导致下游生成的报告哈希不稳定；
- **去重保序合并（Deduplicated Deterministic Reducer）**：按唯一标识去重并按固定规则排序，无论并行分支以什么顺序完成、重试几次，最终合并出的状态完全一致！

### 2. 先写下你的预测
假设 3 个并行分支各自返回 1 条报价，其中分支 B 因超时重试了 1 次（共产生 4 次写入事件），我们在 **24 种不同的完成顺序**下测试：
- 直接覆盖会丢掉几条报价？
- 简单追加会产生几条重复报价？不同完成顺序下会跑出几种不同的最终结果？
- 去重保序合并会丢掉几条？不同完成顺序下会跑出几种结果？


In [ ]:
# [Cell 3: 实验 2 — 并行状态合并函数（Reducer）在乱序与重试下的表现]
import itertools

# 模拟 4 个并发到达的分支增量更新（其中 branch_B 因超时重试发送了两次）
BRANCH_UPDATES = [
    {"branch_id": "branch_A", "airline": "CA101", "price": 1200},
    {"branch_id": "branch_B", "airline": "MU202", "price": 980},
    {"branch_id": "branch_B", "airline": "MU202", "price": 980},  # 重试产生的重复写入
    {"branch_id": "branch_C", "airline": "CZ303", "price": 1050},
]

def reducer_overwrite(current: List[Dict[str, Any]], delta: Dict[str, Any]) -> List[Dict[str, Any]]:
    """反例 1：直接覆盖，只保留最后到达的那条。"""
    return [delta]

def reducer_naive_append(current: List[Dict[str, Any]], delta: Dict[str, Any]) -> List[Dict[str, Any]]:
    """反例 2：无脑追加，不处理重试重复，也不保证顺序稳定。"""
    return current + [delta]

# 🛠️ 作业任务 2：实现去重且顺序稳定的 Reducer（按 branch_id 去重并按 branch_id 升序排列）
def reducer_dedup_sorted(current: List[Dict[str, Any]], delta: Dict[str, Any]) -> List[Dict[str, Any]]:
    """正例：无论分支什么顺序到达、重试几次，结果始终是 3 条且顺序固定。"""
    # === YOUR CODE HERE ===
    merged = {item["branch_id"]: item for item in current}
    merged[delta["branch_id"]] = delta
    return [merged[k] for k in sorted(merged.keys())]
    # ======================

def evaluate_reducer(name: str, reducer_fn: Callable):
    unique_final_states = set()
    counts = []
    for perm in itertools.permutations(BRANCH_UPDATES):
        state: List[Dict[str, Any]] = []
        for update in perm:
            state = reducer_fn(state, update)
        counts.append(len(state))
        unique_final_states.add(json.dumps(state, sort_keys=True))
    print(f"[{name:20s}] 最终保留条数: {counts[0]} 条 (期望 3 条) | 24 种乱序下产生不同结果数: {len(unique_final_states)} 种 (期望 1 种)")

print("=== 实验 2 实测结果（3 分支并发 + 1 次重试，遍历全部 24 种到达顺序）===")
evaluate_reducer("1. 直接覆盖 (Overwrite)", reducer_overwrite)
evaluate_reducer("2. 简单追加 (Append)", reducer_naive_append)
evaluate_reducer("3. 去重保序 (Dedup+Sort)", reducer_dedup_sorted)


---
## 实验 3：回到旧检查点（Time-Travel）——改一处状态，从那里分叉重跑

### 1. 为什么需要它？
在一个 6 步的复杂 Agent 任务中，第 3 步查回的航班预算填错了，直到第 6 步最终核对时才发现。
- 如果没有检查点分叉（Checkpoint Forking），你只能改掉提示词**从第 1 步重头跑 6 步**；
- 如果把每一步的状态都存进 SQLite 检查点表，我们就能直接**回到第 3 步的检查点**，把状态里的错误字段改掉，**只重跑第 4、5、6 步**，甚至从同一个第 3 步检查点分叉出 3 条支路同时试 3 种改法！


In [ ]:
# [Cell 4: 实验 3 — SQLite 检查点时间旅行（Time-Travel & Checkpoint Forking）]

class CheckpointStore:
    """基于内存 SQLite 的轻量级检查点存储，支持按步读取与分叉（Fork）。"""
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute(
            "CREATE TABLE checkpoints (thread_id TEXT, step INT, state_json TEXT, PRIMARY KEY (thread_id, step))"
        )

    def save(self, thread_id: str, step: int, state: Dict[str, Any]) -> None:
        self.conn.execute(
            "INSERT OR REPLACE INTO checkpoints VALUES (?, ?, ?)",
            (thread_id, step, json.dumps(state, ensure_ascii=False)),
        )
        self.conn.commit()

    def load(self, thread_id: str, step: int) -> Dict[str, Any]:
        row = self.conn.execute(
            "SELECT state_json FROM checkpoints WHERE thread_id = ? AND step = ?",
            (thread_id, step),
        ).fetchone()
        if not row:
            raise KeyError(f"Checkpoint not found: {thread_id} step={step}")
        return json.loads(row[0])

    def fork(self, src_thread: str, step: int, new_thread: str, patch: Dict[str, Any]) -> Dict[str, Any]:
        """从 src_thread 的第 step 步检查点克隆状态，打上 patch 修改后存为 new_thread 的起点。"""
        base_state = self.load(src_thread, step)
        forked_state = {**base_state, **patch}
        self.save(new_thread, step, forked_state)
        return forked_state

store = CheckpointStore()

# 1. 初次运行 6 步（第 3 步查出了超预算的航班 2800 元，第 6 步发现超支）
initial_state = {"budget": 2000, "flight_price": 0, "hotel_price": 0, "status": "init"}
for s in range(1, 7):
    if s == 3:
        initial_state["flight_price"] = 2800  # 第 3 步选贵了
    if s == 4:
        initial_state["hotel_price"] = 600
    if s == 6:
        total = initial_state["flight_price"] + initial_state["hotel_price"]
        initial_state["status"] = "OVER_BUDGET" if total > initial_state["budget"] else "OK"
    store.save("main_run", s, initial_state)

# 2. 对比“从头重跑 3 种改法” vs “从第 3 步检查点分叉重跑 3 种改法”
reruns_from_scratch_steps = 3 * 6  # 3 种改法 × 每次重跑全部 6 步 = 18 步
fork_steps = 0
fork_results = {}

for idx, candidate_price in enumerate([1100, 1250, 1350], start=1):
    branch_id = f"fork_branch_{idx}"
    # 直接回到第 3 步检查点改掉 flight_price，只重跑第 4, 5, 6 步（共 3 步）
    st = store.fork("main_run", step=3, new_thread=branch_id, patch={"flight_price": candidate_price})
    for s in (4, 5, 6):
        fork_steps += 1
        if s == 4:
            st["hotel_price"] = 600
        if s == 6:
            total = st["flight_price"] + st["hotel_price"]
            st["status"] = "OVER_BUDGET" if total > st["budget"] else "OK"
        store.save(branch_id, s, st)
    fork_results[branch_id] = st

print("=== 实验 3 实测结果（第 6 步发现第 3 步数据有误，尝试 3 种修正方案）===")
print(f"从头重跑 3 条支路所需总步数      : {reruns_from_scratch_steps} 步")
print(f"从 Step 3 检查点分叉重跑总步数   : {fork_steps} 步 (节省了 {reruns_from_scratch_steps - fork_steps} 步模型调用！)")
for b_id, res in fork_results.items():
    print(f"  -> {b_id}: flight={res['flight_price']}, hotel={res['hotel_price']}, status={res['status']}")


In [ ]:
# [Cell 5: 自动验收与课程回执生成（Grading Receipt）]
assert len(gw_durable.charges) == 100, "实验 1 验收失败：Checkpoint + 幂等键在 100 次崩溃下应严格只产生 100 笔扣款！"
assert durable_calls == 500, "实验 1 验收失败：从 Step 2 检查点恢复应只需 500 次模型调用！"

test_state = []
for u in reversed(BRANCH_UPDATES):
    test_state = reducer_dedup_sorted(test_state, u)
assert len(test_state) == 3 and [x["branch_id"] for x in test_state] == ["branch_A", "branch_B", "branch_C"], "实验 2 验收失败：Reducer 未能正确去重并保序！"
assert fork_steps == 9 and all(r["status"] == "OK" for r in fork_results.values()), "实验 3 验收失败：检查点分叉结果异常！"

receipt = {
    "lesson": "lesson-06-durable-state-graph",
    "status": "PASSED",
    "metrics": {
        "exp1_naive_calls": naive_calls,
        "exp1_durable_calls": durable_calls,
        "exp1_duplicate_charges_prevented": len(gw_naive.charges) - len(gw_durable.charges),
        "exp2_unique_states_across_24_orders": 1,
        "exp3_steps_saved_by_forking": reruns_from_scratch_steps - fork_steps,
    },
}
print("🎉 恭喜！第 6 课全部 3 个实验验收通过！成绩回执如下：")
print(json.dumps(receipt, indent=2, ensure_ascii=False))
